# ISOM 835 · Session 1 — From Data to Decisions: Your First Predictive Model

**Suffolk University · Sawyer Business School · Fall 2026**
Prof. Hasan Arslan · Monday, September 14, 2026 · 5:00–7:30 PM

> Every business question about the future is a prediction problem. Tonight you build a model that answers one — *which customers will leave next month?* — and learn the first three questions to ask of any accuracy number.

**Tonight's loop (you'll repeat it thirteen times this semester):**
frame → data → features → model → evaluate → decide → monitor

Course site: https://isom-835.vercel.app · Session page: https://isom-835.vercel.app/sessions/session-01

In [ ]:
# Environment check — run this cell first. If it fails in Colab: run  !pip install -q -U scikit-learn pandas  then Runtime → Restart session.
import sys, re, sklearn, pandas as pd, numpy as np
need = {'scikit-learn': ('1.6', sklearn.__version__), 'pandas': ('2.2', pd.__version__), 'numpy': ('1.26', np.__version__)}
v = lambda s: tuple(int(x) for x in re.findall(r'\d+', s)[:2])
old = {k: have for k, (want, have) in need.items() if v(have) < v(want)}
assert not old, f'please upgrade {old}: !pip install -q -U ' + ' '.join(old)
print(f'Python {sys.version.split()[0]} ·', ' · '.join(f'{k} {have}' for k, (_, have) in need.items()), '✓')

## 0. Frame the prediction *before* you touch a keyboard

Write these four lines down for every model you ever build. They are the contract with the business.

| | Tonight |
|---|---|
| **Unit** (one row = one what?) | One customer |
| **Target** (what are we predicting?) | `Churn` — did the customer leave? (Yes / No) |
| **Horizon** (how far ahead?) | Next month |
| **Decision** (what changes if we're right?) | Who gets a retention offer |

### Where this sits on the analytics map

| Descriptive | **Predictive** (this course) | Prescriptive (ISOM 839) |
|---|---|---|
| What happened? | What will happen next — and to whom? | What should we do about it? |
| Dashboards, BI | Models that score the future | Optimization over those scores |

A prediction is only valuable when a decision changes because of it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 30)
print('pandas', pd.__version__)

## 1. Load the data

IBM's sample **Telco Customer Churn** dataset: 7,043 customers of a telecom company, 20 features, and a `Churn` column that says whether they left last month. It is hosted in the course repository so it loads from a URL — no login, no upload.

In [ ]:
url = 'https://raw.githubusercontent.com/harslan/isom-835/master/public/data/telco_churn.csv'
df = pd.read_csv(url)

print(df.shape)
df.head()

## 2. Look before you model

Three habits: check the **dtypes** (numbers stored as text are the #1 silent bug), check for **missing values**, and check the **target rate**.

### The `TotalCharges` gotcha

`TotalCharges` should be a number but pandas read it as `object` (text). Eleven brand-new customers have a blank string there. `pd.to_numeric(errors='coerce')` turns the blanks into `NaN` so the column becomes numeric — and we can see exactly who is affected.

In [ ]:
print(df.dtypes)

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

print('Rows with missing TotalCharges:', df['TotalCharges'].isna().sum())
df.loc[df['TotalCharges'].isna(), ['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges', 'Churn']]

All eleven have `tenure == 0` — they joined this month and have not been billed yet. Not an error, just a fact about new customers. For tonight we leave them in (our model does not use `TotalCharges`).

### The target rate — your first baseline

In [ ]:
churn_rate = (df['Churn'] == 'Yes').mean()
print(f'Customers: {len(df):,}')
print(f'Churned last month: {churn_rate:.1%}')
print(f'Stayed:             {1 - churn_rate:.1%}   <- the "nobody churns" baseline')

**26.5% of customers churned.** That means a model that predicts "nobody churns" is right 73.5% of the time. Any model we build has to beat *that* — not zero.

### Is there signal? Churn by contract, tenure, and internet service

If a simple chart shows no difference between groups, no model will find one either.

In [ ]:
by_contract = df.groupby('Contract')['Churn'].apply(lambda s: (s == 'Yes').mean()).sort_values(ascending=False)
print(by_contract.map('{:.1%}'.format))

fig, ax = plt.subplots(figsize=(6, 3.5))
by_contract.plot.bar(ax=ax, color=['#ff6b8b', '#7c8cff', '#2ee6c5'])
ax.set_ylabel('Churn rate'); ax.set_xlabel(''); ax.set_title('Churn rate by contract type')
ax.yaxis.set_major_formatter(lambda v, _: f'{v:.0%}')
plt.xticks(rotation=0); plt.tight_layout(); plt.show()

In [ ]:
df['tenure_bucket'] = pd.cut(df['tenure'], bins=[-1, 6, 12, 24, 48, 72],
                             labels=['0-6 mo', '7-12 mo', '1-2 yr', '2-4 yr', '4-6 yr'])

by_tenure = df.groupby('tenure_bucket', observed=True)['Churn'].apply(lambda s: (s == 'Yes').mean())
by_internet = df.groupby('InternetService')['Churn'].apply(lambda s: (s == 'Yes').mean())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
by_tenure.plot.bar(ax=axes[0], color='#7c8cff'); axes[0].set_title('Churn by tenure'); axes[0].set_xlabel('')
by_internet.plot.bar(ax=axes[1], color='#2ee6c5'); axes[1].set_title('Churn by internet service'); axes[1].set_xlabel('')
for ax in axes:
    ax.yaxis.set_major_formatter(lambda v, _: f'{v:.0%}'); ax.tick_params(axis='x', rotation=0)
plt.tight_layout(); plt.show()

print(by_tenure.map('{:.1%}'.format)); print(); print(by_internet.map('{:.1%}'.format))

Month-to-month customers churn at **43%**; two-year contracts at **3%**. New customers churn far more than long-tenured ones. Fiber-optic customers churn more than DSL. The signal is already visible — the model's job is to combine these signals into one score per customer.

## 3. Split the data — the most important line in the notebook

We hold out 20% of customers the model will **never see** during training. Their score is an honest forecast of how the model will do on *next* month's customers, not a memory of last month's.

`stratify=y` keeps the 26.5% churn rate identical in both halves. `random_state=835` makes your split identical to everyone else's in the room.

In [ ]:
from sklearn.model_selection import train_test_split

X = pd.get_dummies(df[['tenure', 'MonthlyCharges', 'Contract']], drop_first=True)
y = (df['Churn'] == 'Yes').astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=835)

print('Train customers:', len(X_train), '  Test customers:', len(X_test))
print(f'Churn rate — train: {y_train.mean():.1%}   test: {y_test.mean():.1%}')
X.head()

Notice what `pd.get_dummies` did: `Contract` had three text values, so it became two 0/1 columns (`Contract_One year`, `Contract_Two year`; month-to-month is the implied baseline when both are 0). Models need numbers — Session 3 makes this step leak-proof and automatic with pipelines.

## 4. Your first model: twelve lines

Fit on train. Predict on test. Never the other way around.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

model = LogisticRegression(max_iter=1000).fit(X_train, y_train)
pred = model.predict(X_test)

print(f'Baseline (nobody churns): {1 - y_test.mean():.1%}')
print(f'Model accuracy:           {accuracy_score(y_test, pred):.1%}')
print(confusion_matrix(y_test, pred))

**79% vs. 73.5%.** The model beats the do-nothing baseline — but is 79% *good*? That depends on what the mistakes cost, and accuracy cannot tell you. Open the confusion matrix.

## 5. The confusion matrix, in business terms

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

cm = confusion_matrix(y_test, pred)
tn, fp, fn, tp = cm.ravel()

print(f'Loyal customers left alone (true negatives):        {tn}')
print(f'Loyal customers we would bother (false positives):  {fp}')
print(f'Churners we MISSED (false negatives):               {fn}')
print(f'Churners we caught (true positives):                {tp}')
print()
print(f'Share of churners caught (recall):        {tp / (tp + fn):.1%}')
print(f'Share of flagged who really churn (precision): {tp / (tp + fp):.1%}')

ConfusionMatrixDisplay(cm, display_labels=['Stays', 'Churns']).plot(cmap='Blues')
plt.title('Test set: 1,409 customers the model never saw'); plt.show()

The model catches only about **half** of the churners. That bottom-left cell — churners we missed — is where the money leaks. Session 6 is entirely about that cell.

## 6. Scores, not labels: `predict_proba`

`predict()` gave a yes/no label using a 50% cutoff. Underneath, the model produced a **probability** for every customer. Probabilities are what a business actually uses: rank customers, pick the riskiest, decide how far down the list the retention budget reaches.

In [ ]:
proba = model.predict_proba(X_test)[:, 1]

scored = df.loc[X_test.index, ['customerID', 'tenure', 'MonthlyCharges', 'Contract', 'Churn']].copy()
scored['churn_probability'] = proba.round(3)

from sklearn.metrics import roc_auc_score
print(f'ROC-AUC: {roc_auc_score(y_test, proba):.3f}   (0.5 = coin flip, 1.0 = perfect ranking; Session 5 reads this properly)')
print()
print('Ten riskiest customers in the test set:')
scored.sort_values('churn_probability', ascending=False).head(10)

## 7. Why accuracy lies

Three questions to ask of *any* accuracy number:

1. **Compared with what baseline?** 79% sounds good until you hear the do-nothing model gets 73.5%.
2. **On which data?** Training accuracy is a memory. Only held-out accuracy is a forecast.
3. **At what cost per mistake?** Accuracy weighs every error the same. The business does not.

### A simple cost model (assumptions stated)

- A retention offer costs **$50** and we assume it keeps the customer (a generous simplification — Session 6 relaxes it).
- A churner we miss costs **$1,500** in lost lifetime value.
- Every customer we flag gets the offer, whether or not they would have left.

In [ ]:
OFFER_COST = 50
LOST_VALUE = 1500

def cost_of_mistakes(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    offers = (tp + fp) * OFFER_COST          # everyone we flag gets an offer
    losses = fn * LOST_VALUE                 # churners we missed walk away
    return offers, losses, offers + losses

baseline_pred = np.zeros_like(y_test)        # "nobody churns" -> offer nobody
for name, p in [('Baseline (offer nobody)', baseline_pred), ('Model @ 50% threshold', pred)]:
    offers, losses, total = cost_of_mistakes(y_test, p)
    print(f'{name:26s} offers ${offers:>8,}   missed churn ${losses:>9,}   total ${total:>9,}')

Under these assumptions the model already saves the company roughly **$280K** on 1,409 customers — and it is still missing 173 churners. Lowering the threshold (flag anyone above 30% instead of 50%) costs more offers but catches more churners. Whether that is worth it is a *business* question with a *numeric* answer — that is Session 6.

## 8. Preview: more features, same recipe

Tonight's model used three columns. The recipe does not change when you use all twenty — only `X` does.

In [ ]:
feature_cols = df.columns.drop(['customerID', 'Churn', 'tenure_bucket'])
X_all = pd.get_dummies(df[feature_cols], drop_first=True).fillna(0)

Xa_train, Xa_test, ya_train, ya_test = train_test_split(
    X_all, y, test_size=0.2, stratify=y, random_state=835)

model_all = LogisticRegression(max_iter=3000).fit(Xa_train, ya_train)
proba_all = model_all.predict_proba(Xa_test)[:, 1]

print(f'Features: {X_all.shape[1]}')
print(f'Accuracy: {accuracy_score(ya_test, model_all.predict(Xa_test)):.1%}   (3-feature model: {accuracy_score(y_test, pred):.1%})')
print(f'ROC-AUC:  {roc_auc_score(ya_test, proba_all):.3f}   (3-feature model: {roc_auc_score(y_test, proba):.3f})')

More features, slightly better model. (Note: this quick version used `get_dummies` on the whole table *before* splitting. That is harmless here because dummy-coding does not learn anything from the target — but scaling, imputation, and target encoding do, and doing them before the split is the classic **leak**. Session 3 fixes it properly with pipelines.)

## 9. Your turn (in class — and the start of Homework #1)

**Exercise 1 — Add features.** Rebuild the 3-feature model with three more columns of your choice (e.g. `InternetService`, `TechSupport`, `PaymentMethod`). Did accuracy improve? Did the confusion matrix improve — and which cell moved?

In [ ]:
# Exercise 1: edit the column list, re-run.
my_cols = ['tenure', 'MonthlyCharges', 'Contract']  # add 3 more here

X1 = pd.get_dummies(df[my_cols], drop_first=True)
X1_train, X1_test, y1_train, y1_test = train_test_split(X1, y, test_size=0.2, stratify=y, random_state=835)
m1 = LogisticRegression(max_iter=3000).fit(X1_train, y1_train)
print(f'Accuracy: {accuracy_score(y1_test, m1.predict(X1_test)):.1%}')
print(confusion_matrix(y1_test, m1.predict(X1_test)))

**Exercise 2 — Change the threshold.** Flag a customer as a churner when the probability is above **0.30** instead of 0.50. Recompute the confusion matrix and the cost. What happened to the missed-churners cell? To the offers bill?

**Exercise 3 — Where are the misses?** Among the churners the 50%-threshold model missed, which contract type is most common? Does that surprise you given the churn-by-contract chart?

In [ ]:
# Exercise 2: threshold at 0.30
pred_30 = (proba >= 0.30).astype(int)
print(confusion_matrix(y_test, pred_30))
offers, losses, total = cost_of_mistakes(y_test, pred_30)
print(f'Model @ 30% threshold      offers ${offers:>8,}   missed churn ${losses:>9,}   total ${total:>9,}')
print()

# Exercise 3: missed churners by contract type
missed = scored[(scored['Churn'] == 'Yes') & (pred == 0)]
print('Missed churners by contract:'); print(missed['Contract'].value_counts())

## What we learned tonight

- **Predictive analytics** sits between describing the past and prescribing the future. A prediction is only valuable when a decision changes because of it.
- **The loop is always the same:** frame → data → features → model → evaluate → decide → monitor. The framing step (unit, target, horizon, decision) is the one business people skip and the one that matters most.
- **Split first.** Held-out data is the only honest score. `stratify=y`, fixed `random_state`, fit on train, predict on test.
- **Accuracy lies.** Ask: compared with what baseline? on which data? at what cost per mistake? A "nobody churns" model is 73.5% accurate and worth nothing.
- A model produces **scores**; a business produces **decisions**. Thresholds and costs are where the money is (Session 6).

## Before Session 2 — nothing to submit

1. Run every cell in this notebook, then try **Exercise 1**: add three features of your choice and see whether accuracy *and* the confusion matrix improve.
2. **Homework #1** opens in Session 2 (due Mon Sep 28) and starts from exactly this notebook — so the more of it you have touched, the easier it is.
3. Keep an eye out for a **prediction problem from your own work or life** — the unit, the target, the horizon, and the decision it would change. HW1 asks you to frame one, and it may become your final project.

*Next week — Session 2: Data Wrangling & EDA for Prediction. Real data arrives in five tables, three date formats, and a thousand missing values.*